In [1]:
# ==========================================
# BLOQUE 1: PREPARACIÓN Y ENTORNO (PYTORCH GPU)
# ==========================================
import os
import glob
import time
import numpy as np
import librosa
import soundfile as sf
import torch
import torch.nn as nn
import torch.optim as optim
import warnings

# Supresión de warnings innecesarios de audios cortos
warnings.filterwarnings("ignore", category=UserWarning)

print("[INICIALIZANDO SISTEMA USAR - ESTACIÓN ZONA CERO]")

# Verificación crítica de hardware (NVIDIA RTX 3060)
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f"✅ [ÉXITO] Motor de Deep Learning activo. GPU Detectada: {device_name}")
    device = torch.device("cuda")
else:
    print("⚠️ [ADVERTENCIA] No se detectó GPU CUDA. Migrando a CPU (Latencia alta esperada).")
    device = torch.device("cpu")

[INICIALIZANDO SISTEMA USAR - ESTACIÓN ZONA CERO]
✅ [ÉXITO] Motor de Deep Learning activo. GPU Detectada: NVIDIA GeForce RTX 4060


In [2]:
# ==========================================
# BLOQUE 2: PROCESAMIENTO DSP Y EXTRACCIÓN
# ==========================================

# Configuración del hardware HIKMICRO AD21p y segmentación
SAMPLE_RATE = 16000
WINDOW_SECONDS = 3
OVERLAP_SECONDS = 1
HOP_SECONDS = WINDOW_SECONDS - OVERLAP_SECONDS

def get_mel_spec_from_frame(audio_frame):
    """Genera el espectrograma de Mel respetando el filtro 150-7500 Hz"""
    mel_spec = librosa.feature.melspectrogram(
        y=audio_frame, sr=SAMPLE_RATE, n_mels=128,
        fmin=150, fmax=7500, hop_length=512
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    # Normalización Min-Max para los tensores de PyTorch
    mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-6)
    
    # Añadimos canal (1, 128, frames) para la CNN
    return np.expand_dims(mel_spec_db, axis=0) 
# ==========================================
# BLOQUE 2.5: DATA LOADER, PARSEO Y BALANCEO
# ==========================================
import os
import glob
import pandas as pd
import numpy as np
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

print("📡 [DATA ENGINEERING] Iniciando parseo y unificación de orígenes de datos...")

# Rutas base (Ajustar según la estación de mando)
BASE_DIR = r"C:\Users\carlo\Documents\detector_vida_acustico\Dataset\.Datasets_obligatorios"
PATH_ZONA_CERO = os.path.join(BASE_DIR, "dataset_manual")
PATH_VOZ = os.path.join(BASE_DIR, "voz_espanol")
PATH_ESC50 = os.path.join(BASE_DIR, "ESC_50")
PATH_URBAN = os.path.join(BASE_DIR, "UrbanSound8K")

def parse_all_datasets():
    dataset_files = [] # Lista de tuplas: (ruta_archivo, etiqueta)
    
    # ---------------------------------------------------------
    # RUTA 1: Dataset Maestro (Zona Cero)
    # ---------------------------------------------------------
    if os.path.exists(PATH_ZONA_CERO):
        for subfolder in os.listdir(PATH_ZONA_CERO):
            sub_path = os.path.join(PATH_ZONA_CERO, subfolder)
            if not os.path.isdir(sub_path): continue
            
            # Target = 1, Ruido = 0
            label = 1.0 if subfolder in ["Target_Voice", "Target_Rhythmic"] else 0.0
            for file in glob.glob(os.path.join(sub_path, "*.wav")):
                dataset_files.append((file, label))

    # ---------------------------------------------------------
    # RUTA 2: Refuerzo Vida (voz_espanol) -> Etiqueta 1
    # ---------------------------------------------------------
    if os.path.exists(PATH_VOZ):
        
        for file in glob.glob(os.path.join(PATH_VOZ, "*.mp3")): # O .wav si ya fue convertido
            
            dataset_files.append((file, 1.0))

    # ---------------------------------------------------------
    # RUTA 3: Refuerzo Ruido (ESC_50) -> Etiqueta 0
    # ---------------------------------------------------------
    if os.path.exists(PATH_ESC50):
        for file in glob.glob(os.path.join(PATH_ESC50, "*.wav")):
            dataset_files.append((file, 0.0))

    # ---------------------------------------------------------
    # RUTA 4: Refuerzo Ruido Urbano (UrbanSound8K) -> Etiqueta 0
    # ---------------------------------------------------------
    csv_path = os.path.join(PATH_URBAN, "metadata", "UrbanSound8K.csv")
    if os.path.exists(csv_path):
        df_urban = pd.read_csv(csv_path)
        
        clases_prohibidas = [2, 9]
        df_urban_filtered = df_urban[~df_urban['classID'].isin(clases_prohibidas)]
        
        for index, row in df_urban_filtered.iterrows():
            fold = f"fold{row['fold']}"
            file_name = row['slice_file_name']
            file_path = os.path.join(PATH_URBAN, "audio", fold, file_name)
            if os.path.exists(file_path):
                dataset_files.append((file_path, 0.0))
                
    return dataset_files

# Ejecutar parseo
all_data = parse_all_datasets()
df_master = pd.DataFrame(all_data, columns=['filepath', 'label'])

# ==========================================
# CALCULO MATEMÁTICO DEL PESO POSICIONAL (pos_weight)
# ==========================================
num_clase_0 = len(df_master[df_master['label'] == 0.0])
num_clase_1 = len(df_master[df_master['label'] == 1.0])

# pos_weight = (Cantidad de Negativos) / (Cantidad de Positivos)
# Esto obliga a la CNN a penalizar severamente si falla al detectar la escasez de "Señales de Vida"
pos_weight_value = num_clase_0 / max(num_clase_1, 1)
pos_weight_tensor = torch.tensor([pos_weight_value], dtype=torch.float32).cuda()

print(f"✅ [ESTADÍSTICAS] Ruido (0): {num_clase_0} | Vida (1): {num_clase_1}")
print(f"⚖️ [BALANCEO] pos_weight calculado: {pos_weight_value:.4f}")

# ==========================================
# DEFINICIÓN DEL DATASET DE PYTORCH (CON DEBUGGING Y CARGA ROBUSTA)
# ==========================================
class USARSensorDataset(Dataset):
    def __init__(self, dataframe, sr=16000, duration=3.0):
        self.dataframe = dataframe
        self.sr = sr
        self.max_len = int(sr * duration)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        file_path = self.dataframe.iloc[idx]['filepath']
        label = self.dataframe.iloc[idx]['label']

        try:
            # 1. Carga directa con soundfile (Mejor compatibilidad para WAVs manuales)
            y, orig_sr = sf.read(file_path)
            
            # 2. Corrección de canales: Si el editor lo guardó en estéreo, forzar a mono
            if len(y.shape) > 1:
                y = np.mean(y, axis=1)
                
            # 3. Remuestreo automático si la tasa de origen no es 16000 Hz
            if orig_sr != self.sr:
                y = librosa.resample(y, orig_sr=orig_sr, target_sr=self.sr)

            # 4. Estandarización a 3 segundos (Padding o Truncating)
            if len(y) > self.max_len:
                y = y[:self.max_len]
            else:
                y = np.pad(y, (0, self.max_len - len(y)), 'constant')

            # 5. Extracción del Espectrograma Mel
            mel_spec = librosa.feature.melspectrogram(y=y, sr=self.sr, n_mels=128, fmin=150, fmax=7500)
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
            mel_spec_db = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min() + 1e-6)

            tensor_x = torch.tensor(mel_spec_db, dtype=torch.float32).unsqueeze(0)
            tensor_y = torch.tensor([label], dtype=torch.float32)

            return tensor_x, tensor_y

        except Exception as e:
            # ELIMINAMOS LA FALLA SILENCIOSA: Ahora te gritará en consola si falla
            print(f"\n❌ [ERROR CRÍTICO LEYENDO ARCHIVO]: {file_path}")
            print(f"👉 Detalle del sistema: {e}")
            
            # Retornamos ceros para no colapsar la GPU, pero mantenemos tu etiqueta original
            return torch.zeros((1, 128, int(self.max_len/512)+1)), torch.tensor([label], dtype=torch.float32)

# ==========================================
# CREACIÓN DE DATALOADERS (SPLIT 80/20)
# ==========================================
train_df, val_df = train_test_split(df_master, test_size=0.20, stratify=df_master['label'], random_state=42)

train_dataset = USARSensorDataset(train_df)
val_dataset = USARSensorDataset(val_df)

# DataLoader optimizado (Windows Safe)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0, pin_memory=True)

print("🚀 [SISTEMA LISTO] DataLoaders generados. Preparado para entrenar en la GPU con BCEWithLogitsLoss.")
# NOTA PARA EL ENTRENAMIENTO:
# criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

📡 [DATA ENGINEERING] Iniciando parseo y unificación de orígenes de datos...
✅ [ESTADÍSTICAS] Ruido (0): 7240 | Vida (1): 5009
⚖️ [BALANCEO] pos_weight calculado: 1.4454
🚀 [SISTEMA LISTO] DataLoaders generados. Preparado para entrenar en la GPU con BCEWithLogitsLoss.


In [3]:
# ==========================================
# BLOQUE 3: ARQUITECTURA CNN Y ENTRENAMIENTO
# ==========================================

class GaussianNoise(nn.Module):
    """Capa personalizada para evitar Overfitting inyectando estática simulada"""
    def __init__(self, std=0.1):
        super().__init__()
        self.std = std

    def forward(self, x):
        if self.training and self.std > 0:
            noise = torch.randn_like(x) * self.std
            return x + noise
        return x

class USAR_CNN(nn.Module):
    def __init__(self):
        super(USAR_CNN, self).__init__()
        # Inyección crítica anti-overfitting
        self.noise = GaussianNoise(std=0.1)
        
        # Extracción de características espaciales del audio
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        # Clasificador
        self.fc_block = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128 * 16 * 11, 128), # Ajustado para entrada 128x94
            nn.ReLU(),
            nn.Linear(128, 1) # Salida binaria (Sin activación, usaremos BCEWithLogitsLoss)
        )

    def forward(self, x):
        x = self.noise(x)
        x = self.conv_block(x)
        x = x.view(x.size(0), -1) # Flatten
        x = self.fc_block(x)
        return x

# Instanciar modelo y enviarlo a la GPU
usar_model = USAR_CNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(usar_model.parameters(), lr=1e-4)

print("[INFO] Arquitectura CNN cargada en la GPU con inyección de ruido.")
# (Aquí iría tu bucle de entrenamiento normal con tus datos etiquetados)


# ==========================================
# BLOQUE 3.5: MOTOR DE ENTRENAMIENTO (TRAINING LOOP)
# ==========================================
import time
from sklearn.metrics import roc_auc_score

EPOCHS = 20
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
# El optimizador 'optimizer' ya fue definido en el Bloque 3

print("[INICIANDO COMPILACIÓN EN RTX 4060] Iniciando 20 Épocas...")
start_time = time.time()

for epoch in range(EPOCHS):
    # --- FASE DE ENTRENAMIENTO ---
    usar_model.train() # Activa Dropouts y el GaussianNoise
    train_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device).float().view(-1, 1)
        
        optimizer.zero_grad()
        outputs = usar_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # --- FASE DE VALIDACIÓN ---
    usar_model.eval() # Apaga Dropouts y ruido para evaluar fríamente
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device).float().view(-1, 1)
            outputs = usar_model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Guardar predicciones para el cálculo del AUC
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(probs)
            all_targets.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader)
    
    # Calcular métrica AUC (Área bajo la curva ROC)
    # Un AUC > 0.85 indica un modelo listo para rescate
    auc_score = roc_auc_score(all_targets, all_preds)
    
    print(f"Época [{epoch+1}/{EPOCHS}] | Pérdida Train: {avg_train_loss:.4f} | Pérdida Val: {avg_val_loss:.4f} | Precisión (AUC): {auc_score:.4f}")

elapsed_time = (time.time() - start_time) / 60
print(f"\n[ENTRENAMIENTO COMPLETADO] Tiempo total: {elapsed_time:.2f} minutos.")



# ==========================================
# BLOQUE 3.6: EXPORTACIÓN TÁCTICA DEL MODELO
# ==========================================
import torch
import os

# Definimos el nombre del archivo del modelo entrenado
MODEL_FILENAME = "usar_cnn_pesos_zona_cero.pth"

print("💾 [INICIANDO EXPORTACIÓN]")

# Guardamos EXCLUSIVAMENTE los pesos (state_dict) - Máxima compatibilidad
torch.save(usar_model.state_dict(), MODEL_FILENAME)

ruta_absoluta = os.path.abspath(MODEL_FILENAME)
print(f"✅ [MODELO EXPORTADO CON ÉXITO] Archivo generado: {MODEL_FILENAME}")
print(f"📍 Ubicación: {ruta_absoluta}")
print("\n>>> ACCIÓN REQUERIDA: Copia este archivo '.pth' a un pendrive y transfiérelo a tu laptop de campo.")

[INFO] Arquitectura CNN cargada en la GPU con inyección de ruido.
[INICIANDO COMPILACIÓN EN RTX 4060] Iniciando 20 Épocas...
Época [1/20] | Pérdida Train: 0.1140 | Pérdida Val: 0.0661 | Precisión (AUC): 0.9986
Época [2/20] | Pérdida Train: 0.0558 | Pérdida Val: 0.0884 | Precisión (AUC): 0.9988
Época [3/20] | Pérdida Train: 0.0388 | Pérdida Val: 0.0672 | Precisión (AUC): 0.9986
Época [4/20] | Pérdida Train: 0.0308 | Pérdida Val: 0.0764 | Precisión (AUC): 0.9984
Época [5/20] | Pérdida Train: 0.0283 | Pérdida Val: 0.0734 | Precisión (AUC): 0.9982
Época [6/20] | Pérdida Train: 0.0270 | Pérdida Val: 0.0976 | Precisión (AUC): 0.9978
Época [7/20] | Pérdida Train: 0.0253 | Pérdida Val: 0.0746 | Precisión (AUC): 0.9980
Época [8/20] | Pérdida Train: 0.0196 | Pérdida Val: 0.0794 | Precisión (AUC): 0.9986
Época [9/20] | Pérdida Train: 0.0189 | Pérdida Val: 0.0491 | Precisión (AUC): 0.9991
Época [10/20] | Pérdida Train: 0.0223 | Pérdida Val: 0.0539 | Precisión (AUC): 0.9989
Época [11/20] | Pérdida 

In [4]:
# ==========================================
# BLOQUE 4: INFERENCIA OFFLINE POR LOTES (GPU)
# ==========================================

def batch_analyze_folder(folder_path, model):
    print(f"\n📡 [MODO BATCH ACTIVADO] Escaneando directorio: {folder_path}")
    wav_files = sorted(glob.glob(os.path.join(folder_path, "*.wav")))

    if not wav_files:
        print("⚠️ No se encontraron archivos de audio.")
        return

    model.eval() # Modo inferencia (apaga el Dropout y el GaussianNoise)

    with torch.no_grad(): # Desactiva el cálculo de gradientes (Ahorra muchísima VRAM)
        for file_path in wav_files:
            filename = os.path.basename(file_path)
            try:
                # 1. Ingesta del archivo (hasta 1 min)
                audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
                frame_length = WINDOW_SECONDS * sr
                hop_length = HOP_SECONDS * sr

                if len(audio) < frame_length:
                    padding = frame_length - len(audio)
                    audio = np.pad(audio, (0, padding), 'constant')

                # 2. Segmentación en ventanas (Solapamiento)
                frames = librosa.util.frame(audio, frame_length=frame_length, hop_length=hop_length).T

                # 3. Procesamiento a Tensor Batch
                batch_features = [get_mel_spec_from_frame(frame) for frame in frames]
                
                # Forma del tensor: (batch_size, channels=1, mel_bins=128, time_steps)
                batch_tensor = torch.tensor(np.array(batch_features), dtype=torch.float32).to(device)

                # 4. Inferencia Masiva en GPU
                outputs = model(batch_tensor)
                probabilities = torch.sigmoid(outputs).cpu().numpy().flatten() * 100

                # 5. Reporte Táctico
                alert = False
                for i, prob in enumerate(probabilities):
                    if prob > 80.0: # Umbral alto por seguridad
                        start_t = i * HOP_SECONDS
                        end_t = start_t + WINDOW_SECONDS
                        print(f"🚨 [ALERTA] {filename} -> SEÑAL DE VIDA DETECTADA ({prob:.1f}%) | Rango: {start_t}s - {end_t}s")
                        alert = True
                
                if not alert:
                    print(f"✅ {filename} -> Despejado (Ruido de fondo).")

            except Exception as e:
                print(f"❌ Error en {filename}: {e}")

    print("\n[ESCANEO FINALIZADO] Revisar marcas de tiempo para despliegue de brigada.")


In [5]:
# ==========================================
# EJECUCIÓN TÁCTICA: BARRIDO DE ZONA CERO
# ==========================================
import os

# Ruta exacta de la memoria SD del sensor HIKMICRO AD21p
ruta_operacion = r"C:\Users\carlo\Documents\detector_vida_acustico\Dataset\1manual\AD21PVenimeca\DCIM\202607"

print(f"⚙️ [SISTEMA] Verificando acceso al directorio: {ruta_operacion}")

if os.path.exists(ruta_operacion):
    # Ejecutamos la función de Inferencia Batch (Bloque 4)
    # Nota: Si estás en la laptop de campo, usa 'modelo_despliegue' en lugar de 'usar_model'
    batch_analyze_folder(ruta_operacion, usar_model)
else:
    print("❌ [ERROR] El sistema no encuentra la ruta. Verifica si el AD21p está conectado o si la ruta está bien escrita.")

⚙️ [SISTEMA] Verificando acceso al directorio: C:\Users\carlo\Documents\detector_vida_acustico\Dataset\1manual\AD21PVenimeca\DCIM\202607

📡 [MODO BATCH ACTIVADO] Escaneando directorio: C:\Users\carlo\Documents\detector_vida_acustico\Dataset\1manual\AD21PVenimeca\DCIM\202607
✅ HM20260705034457_5_process.wav -> Despejado (Ruido de fondo).
🚨 [ALERTA] HM20260705192416_9_process.wav -> SEÑAL DE VIDA DETECTADA (88.0%) | Rango: 2s - 5s
🚨 [ALERTA] HM20260705192559_9_process.wav -> SEÑAL DE VIDA DETECTADA (81.6%) | Rango: 40s - 43s
🚨 [ALERTA] HM20260705192559_9_process.wav -> SEÑAL DE VIDA DETECTADA (93.6%) | Rango: 46s - 49s
🚨 [ALERTA] HM20260705192559_9_process.wav -> SEÑAL DE VIDA DETECTADA (90.8%) | Rango: 52s - 55s
🚨 [ALERTA] HM20260705192651_9_process.wav -> SEÑAL DE VIDA DETECTADA (93.0%) | Rango: 20s - 23s
🚨 [ALERTA] HM20260705192651_9_process.wav -> SEÑAL DE VIDA DETECTADA (85.8%) | Rango: 22s - 25s
🚨 [ALERTA] HM20260705192651_9_process.wav -> SEÑAL DE VIDA DETECTADA (94.5%) | Rango: 2